# Paper 11 · BERT

**Citation:** Jacob Devlin et al., “BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding” (2018).

**Paper:** https://arxiv.org/abs/1810.04805

> **Scale gap:** We reproduce the value of bidirectional context on a toy masked-token task, not BERT pretraining.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 11 · Attention & Transformer Mathematics](../../math/11_attention_transformers.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. What information can masked-language modeling use that causal next-token prediction cannot?
2. What exactly is transferred during fine-tuning?
3. Why was pretraining plus fine-tuning important?

## Central claim
Deep bidirectional Transformer representations can be pretrained from unlabeled text and then fine-tuned for downstream tasks.

## Toy masked-token problem
The hidden middle bit is XOR(left, right). A model that sees only the left context cannot solve it above chance.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-11_bert', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/11_bert.ipynb')
experiment.capture_figures()

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0)
N=4000
left=torch.randint(0,2,(N,)); right=torch.randint(0,2,(N,))
target=left^right
idx=torch.randperm(N); tr=idx[:3000]; te=idx[3000:]

class ContextModel(nn.Module):
    def __init__(self,bidirectional=True):
        super().__init__(); self.bidirectional=bidirectional
        d=8; self.emb=nn.Embedding(2,d)
        self.net=nn.Sequential(nn.Linear(d*(2 if bidirectional else 1),16),nn.ReLU(),nn.Linear(16,2))
    def forward(self,l,r):
        parts=[self.emb(l)]
        if self.bidirectional: parts.append(self.emb(r))
        return self.net(torch.cat(parts,1))

def train(bidir):
    torch.manual_seed(0); m=ContextModel(bidir); opt=torch.optim.Adam(m.parameters(),lr=.02); ce=nn.CrossEntropyLoss(); hist=[]
    for _ in range(120):
        opt.zero_grad(); loss=ce(m(left[tr],right[tr]),target[tr]); loss.backward(); opt.step()
        with torch.no_grad(): acc=(m(left[te],right[te]).argmax(1)==target[te]).float().mean().item()
        hist.append(acc)
    return np.array(hist)
b=train(True); c=train(False)
print("bidirectional",b[-1],"left-only causal proxy",c[-1])
plt.plot(b,label="both sides"); plt.plot(c,label="left only"); plt.legend(); plt.show()

## Optional Hugging Face architecture inspection

In [ ]:
import importlib.util
if importlib.util.find_spec('transformers') is not None:
    from transformers import BertConfig, BertForMaskedLM
    cfg=BertConfig(vocab_size=100,hidden_size=32,num_hidden_layers=2,num_attention_heads=4,intermediate_size=64)
    m=BertForMaskedLM(cfg)
    ids=torch.randint(0,100,(2,10))
    print("random-weight BERT MLM logits:",m(ids).logits.shape)
else:
    print('NOT RUN: optional Transformers architecture inspection requires its package.')

### Ablation
Mask different fractions of inputs in a toy reconstruction task. Explain the tradeoff between too little and too much corruption.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))

## Methodology note

This is a bounded mechanism experiment, not an original-benchmark reproduction. A run that completes is not evidence that the paper claim was reproduced. Report actual baseline comparisons, uncertainty and failed ablations.